## SerpAPI Search Results Test
Testing what raw search results look like from both search implementations.

In [1]:
import os
from dotenv import load_dotenv
from serpapi import GoogleSearch

load_dotenv()
print("API key set:", bool(os.getenv("SERPAPI_API_KEY")))

API key set: True


### Test 1: Primary search node (5 results)
Mimics `graph/nodes/search.py` — used for listing discovery.

In [3]:
import json

query = "2 bedroom apartments for rent San Francisco site:zillow.com inurl:homedetails"

search = GoogleSearch({
    "q": query,
    "api_key": os.getenv("SERPAPI_API_KEY"),
    "num": 5,
})

raw = search.get_dict()
organic = raw.get("organic_results", [])

results = []
for r in organic[:5]:
    results.append({
        "title": r.get("title", ""),
        "link": r.get("link", ""),
        "snippet": r.get("snippet", ""),
        "source": r.get("source", ""),
    })

print(json.dumps({"query": query, "results": results}, indent=2))

{
  "query": "2 bedroom apartments for rent San Francisco site:zillow.com inurl:homedetails",
  "results": [
    {
      "title": "538 3rd Ave APT 2, San Francisco, CA 94118",
      "link": "https://www.zillow.com/homedetails/538-3rd-Ave-APT-2-San-Francisco-CA-94118/461374502_zpid/",
      "snippet": "538 3rd Ave APT 2, San Francisco, CA 94118 is an apartment unit listed for rent at $3100 /mo. The -- sqft unit is a 1 bed, 1 bath apartment ...",
      "source": "Zillow"
    },
    {
      "title": "2221 Pacific Ave APT 2, San Francisco, CA 94115",
      "link": "https://www.zillow.com/homedetails/2221-Pacific-Ave-APT-2-San-Francisco-CA-94115/459095827_zpid/",
      "snippet": "OPEN HOUSE: Sunday (3/29) 1:15pm-1:45pm 2BR/2BA * $6,600/month annual lease * Newly remodeled interior * Hardwood floors * Stainless steel appliances ...",
      "source": "Zillow"
    },
    {
      "title": "365 2nd Ave APT 4, San Francisco, CA 94118",
      "link": "https://www.zillow.com/homedetails/365-2nd-Av

### Test 2: Fallback search tool (3 results)
Mimics `graph/tools/search.py` — used by listing agents when scraping is insufficient.

In [6]:
query = "rumi at king reddit" # "2099 Martin luther king jr way berkeley pet policy apartments"

search = GoogleSearch({
    "q": query,
    "api_key": os.getenv("SERPAPI_API_KEY"),
    "num": 3,
})

raw = search.get_dict()
results = []
for r in raw.get("organic_results", [])[:3]:
    results.append({
        "title": r.get("title", ""),
        "snippet": r.get("snippet", ""),
        "link": r.get("link", ""),
    })

print(json.dumps(results, indent=2))

[
  {
    "title": "RUMI@King : r/berkeley",
    "snippet": "Anyone got any experience renting with them? The prices seem a bit too good to be true so I want to know if there's anything I should be worried about.",
    "link": "https://www.reddit.com/r/berkeley/comments/1h4n28z/rumiking/"
  },
  {
    "title": "LIST OF PLACES YOU SHOULD !NOT! RENT : r/berkeley",
    "snippet": "Please stay away from renting with K&S, their service is horrible and they do not maintain any of their properties. We have brought up so many ...",
    "link": "https://www.reddit.com/r/berkeley/comments/138y8hn/list_of_places_you_should_not_rent/"
  },
  {
    "title": "HELP WITH SG REAL ESTATE : r/berkeley",
    "snippet": "It's the new RUMI @ King building on MLK right across the street from Trader Joe's ... Hey, im not OP, but I ended up leasing with RUMI for next ...",
    "link": "https://www.reddit.com/r/berkeley/comments/1daq2hc/help_with_sg_real_estate/"
  }
]


### Test 3: Raw SerpAPI response (full)
See everything SerpAPI returns before we strip it down.

In [7]:
query = "1 bedroom apartment for rent Oakland CA"

search = GoogleSearch({
    "q": query,
    "api_key": os.getenv("SERPAPI_API_KEY"),
    "num": 3,
})

raw = search.get_dict()

# Print just the first organic result unstripped so we can see what fields are available
if raw.get("organic_results"):
    print("First raw organic result:")
    print(json.dumps(raw["organic_results"][0], indent=2))
else:
    print("No organic results. Full response:")
    print(json.dumps(raw, indent=2))

First raw organic result:
{
  "position": 1,
  "title": "1 Bedroom Apartments For Rent in Oakland CA",
  "link": "https://www.zillow.com/oakland-ca/apartments-1-bedrooms/",
  "redirect_link": "https://www.google.com/url?sa=t&source=web&rct=j&opi=89978449&url=https://www.zillow.com/oakland-ca/apartments-1-bedrooms/&ved=2ahUKEwj3xYihuMuTAxX7MEQIHYUtE3wQFnoECB8QAQ",
  "displayed_link": "https://www.zillow.com \u203a ... \u203a Alameda County",
  "favicon": "https://serpapi.com/searches/69cc6c98a204ab005f3700c7/images/HhJjaOhflsHe2QBjbU3E7cS9ECm3dIsvSIXtSCwjvuY.png",
  "snippet": "Find your next 1 bedroom apartment in Oakland CA on Zillow. Use our detailed filters to find the perfect place, then get in touch with the property manager.",
  "snippet_highlighted_words": [
    "1 bedroom apartment in Oakland CA on Zillow"
  ],
  "about_this_result": {
    "source": {
      "description": "Find your next 1 bedroom apartment in Oakland CA on Zillow. Use our detailed filters to find the perfect p

### Test 4: Firecrawl scraper
Takes a Zillow listing URL from Test 1 and scrapes the full page content.

In [2]:
from firecrawl import Firecrawl

# Paste a Zillow listing URL from Test 1 results here
url = "https://www.zillow.com/homedetails/538-3rd-Ave-APT-2-San-Francisco-CA-94118/461374502_zpid/"

app = Firecrawl(api_key=os.getenv("FIRECRAWL_API_KEY"))
result = app.scrape(url, formats=["markdown"])

print(result.markdown[:3000])  # print first 3000 chars to see what we get

[Skip main navigation](https://www.zillow.com/homedetails/538-3rd-Ave-APT-2-San-Francisco-CA-94118/461374502_zpid/#skip-topnav-target)

OverviewFacts & featuresPrice historyNeighborhoodCost calculator

Apartment for rent

See all 7 photos

![1st image of 538 3rd Ave APT 2](https://photos.zillowstatic.com/fp/4f74ec89eba08c03cfbad227ab82a699-cc_ft_960.jpg)
![2nd image of 538 3rd Ave APT 2](https://photos.zillowstatic.com/fp/2c09d10973b57fd3482cbb8e5bfa3037-cc_ft_576.jpg)
![3rd image of 538 3rd Ave APT 2](https://photos.zillowstatic.com/fp/985d3b92a794b5472dd0458ea276a3a1-cc_ft_576.jpg)
![4th image of 538 3rd Ave APT 2](https://photos.zillowstatic.com/fp/54697e7c5eb1195baf280b094e663eea-cc_ft_576.jpg)
![5th image of 538 3rd Ave APT 2](https://photos.zillowstatic.com/fp/9f2f5bfaf1c393f7c6d7246226b9b838-cc_ft_576.jpg)

Accepts Zillow applications

$3,100/mo

Fees may apply

# 538 3rd Ave APT 2, San Francisco, CA 94118

1beds

1baths

--sqft

Price may not include required fees and charges. 

### Test 5: Extract structured data from scraped markdown using Claude

In [3]:
import anthropic
import json

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# reuse the markdown from Test 4
listing_markdown = result.markdown

message = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": f"""Extract structured data from this rental listing. Return a JSON object with these fields:
- address (string)
- price_per_month (number, just the digits)
- bedrooms (number)
- bathrooms (number)
- sqft (number or null if not listed)
- pet_policy (string: "allowed", "not allowed", or "unknown")
- parking (string or null)
- laundry (string or null)
- available (string or null)
- lease_term (string or null)

Return only valid JSON, no explanation.

Listing:
{listing_markdown}"""
        }
    ]
)

print("Raw response:", repr(message.content[0].text))

text = message.content[0].text.strip()
# strip markdown code fences if present
if text.startswith("```"):
    text = text.split("```")[1]
    if text.startswith("json"):
        text = text[4:]
    text = text.strip()

structured = json.loads(text)
print(json.dumps(structured, indent=2))

Raw response: '```json\n{\n  "address": "538 3rd Ave APT 2, San Francisco, CA 94118",\n  "price_per_month": 3100,\n  "bedrooms": 1,\n  "bathrooms": 1,\n  "sqft": null,\n  "pet_policy": "not allowed",\n  "parking": "Off street parking",\n  "laundry": "Shared laundry",\n  "available": "now",\n  "lease_term": "1 Year"\n}\n```'
{
  "address": "538 3rd Ave APT 2, San Francisco, CA 94118",
  "price_per_month": 3100,
  "bedrooms": 1,
  "bathrooms": 1,
  "sqft": null,
  "pet_policy": "not allowed",
  "parking": "Off street parking",
  "laundry": "Shared laundry",
  "available": "now",
  "lease_term": "1 Year"
}


### Test 6: Analyze listing photos with Claude vision

In [4]:
import re

# extract image URLs from the markdown returned by Firecrawl
image_urls = re.findall(r'!\[.*?\]\((https://[^)]+)\)', result.markdown)
print(f"Found {len(image_urls)} images:")
for url in image_urls:
    print(" ", url)

# pass images to Claude vision for analysis
message = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": [
                *[
                    {"type": "image", "source": {"type": "url", "url": url}}
                    for url in image_urls[:5]  # cap at 5 images
                ],
                {
                    "type": "text",
                    "text": """Analyze these rental listing photos and return a JSON object with:
- modern_finishes (true/false): updated kitchen/bath, stainless appliances, hardwood floors
- natural_light (true/false): bright, well-lit rooms
- good_views (true/false): any notable view from windows
- spacious (true/false): rooms look reasonably sized
- condition (string): "excellent", "good", "fair", or "poor"
- notes (string): one sentence summary of standout positives or negatives

Return only valid JSON."""
                }
            ]
        }
    ]
)

text = message.content[0].text.strip()
if text.startswith("```"):
    text = text.split("```")[1]
    if text.startswith("json"):
        text = text[4:]
    text = text.strip()

photo_analysis = json.loads(text)
print(json.dumps(photo_analysis, indent=2))

Found 13 images:
  https://photos.zillowstatic.com/fp/4f74ec89eba08c03cfbad227ab82a699-cc_ft_960.jpg
  https://photos.zillowstatic.com/fp/2c09d10973b57fd3482cbb8e5bfa3037-cc_ft_576.jpg
  https://photos.zillowstatic.com/fp/985d3b92a794b5472dd0458ea276a3a1-cc_ft_576.jpg
  https://photos.zillowstatic.com/fp/54697e7c5eb1195baf280b094e663eea-cc_ft_576.jpg
  https://photos.zillowstatic.com/fp/9f2f5bfaf1c393f7c6d7246226b9b838-cc_ft_576.jpg
  https://photos.zillowstatic.com/fp/ce9cfbedbd7896fc7ddf53a6c1ac70a2-p_i.jpg
  https://photos.zillowstatic.com/fp/cc56802675a72fa05a9ac28e8ba344d0-p_i.jpg
  https://photos.zillowstatic.com/fp/359e9d2bd14cfb815d932d92015aa9cb-p_i.jpg
  https://photos.zillowstatic.com/fp/0fa3ec0153ea7aef3ccf24b39591b101-p_i.jpg
  https://photos.zillowstatic.com/fp/3f3992e184fc829a807f99363de022e9-p_c.jpg
  https://photos.zillowstatic.com/fp/1d79e3e54b71625e345275f2b4bc6412-p_c.jpg
  https://photos.zillowstatic.com/fp/3efb177f4012e31412771dd1631a63d6-p_c.jpg
  https://photos.